# pyparsing Package Testing

This notebook tests the pyparsing package - Text parsing library.

## Example 1: Basic text parsing and grammar definition


In [ ]:
import pyparsing as pp

# Example 1: Basic text parsing and grammar definition
print("Example 1: Basic text parsing and grammar definition")

try:
    # Define a simple grammar for parsing numbers and operations
    number = pp.Word(pp.nums)
    operator = pp.oneOf("+ - * /")
    
    # Simple arithmetic expression: number operator number
    expression = number + operator + number
    
    # Test parsing: every input is valid for `number operator number`, so a ParseException fails the cell
    test_strings = {"5 + 3": ["5", "+", "3"], "10 * 2": ["10", "*", "2"], "15 - 7": ["15", "-", "7"], "20 / 4": ["20", "/", "4"]}
    
    for test_str, expected in test_strings.items():
        result = expression.parseString(test_str)
        assert list(result) == expected, (test_str, list(result))
        print(f"Parsed '{test_str}': {list(result)}")
    
    # Parse with results names for easier access
    named_expression = number("left") + operator("op") + number("right")
    
    result = named_expression.parseString("42 + 8")
    assert (result.left, result.op, result.right) == ("42", "+", "8"), result.dump()
    print(f"Named parsing: left={result.left}, op={result.op}, right={result.right}")
    
    print("Basic text parsing and grammar definition completed successfully")
    
except ImportError as e:
    print(f"Import error: {e}")
except Exception as e:
    print(f"Runtime error: {e}")
    raise AssertionError(
        f"pyparsing example (cell 2) failed: {type(e).__name__}: {e}"
    ) from e

print("Example 1 finished")

## Example 2: Complex grammar with nested structures


In [ ]:
import pyparsing as pp

# Example 2: Complex grammar with nested structures
print("Example 2: Complex grammar with nested structures")

try:
    # Define grammar for parsing simple JSON-like structures
    integer = pp.Word(pp.nums).setParseAction(lambda t: int(t[0]))
    string = pp.QuotedString('"')
    
    # Forward declaration for recursive structures
    value = pp.Forward()
    
    # Array definition
    array = pp.Group(pp.Suppress('[') + 
                    pp.Optional(value + pp.ZeroOrMore(pp.Suppress(',') + value)) +
                    pp.Suppress(']'))
    
    # Object definition
    pair = pp.Group(string + pp.Suppress(':') + value)
    obj = pp.Group(pp.Suppress('{') +
                  pp.Optional(pair + pp.ZeroOrMore(pp.Suppress(',') + pair)) +
                  pp.Suppress('}'))
    
    # Define what a value can be
    value <<= integer | string | array | obj
    
    # Test parsing JSON-like structures
    test_cases = [
        '42',
        '"hello"',
        '[1, 2, 3]',
        '{"name": "John", "age": 30}',
        '[{"id": 1, "tags": ["python", "parsing"]}, {"id": 2, "tags": ["json"]}]'
    ]
    
    # What each case parses to: ints converted by the parse action, strings unquoted,
    # arrays and objects as nested lists. Every case is valid, so a ParseException fails the cell
    expected_results = [
        [42],
        ["hello"],
        [[1, 2, 3]],
        [[["name", "John"], ["age", 30]]],
        [[[["id", 1], ["tags", ["python", "parsing"]]], [["id", 2], ["tags", ["json"]]]]],
    ]
    
    for test_case, expected in zip(test_cases, expected_results, strict=True):
        result = value.parseString(test_case)
        assert result.asList() == expected, (test_case, result.asList())
        print(f"Parsed '{test_case}': {result.asList()}")
    
    print("Complex grammar with nested structures completed successfully")
    
except ImportError as e:
    print(f"Import error: {e}")
except Exception as e:
    print(f"Runtime error: {e}")
    raise AssertionError(
        f"pyparsing example (cell 4) failed: {type(e).__name__}: {e}"
    ) from e

print("Example 2 finished")

## Example 3: Parse actions and transformations


In [ ]:
import pyparsing as pp

# Example 3: Parse actions and transformations
print("Example 3: Parse actions and transformations")

try:
    # Define grammar with parse actions for immediate evaluation
    def evaluate_expr(tokens):
        """Evaluate a simple arithmetic expression."""
        left, op, right = tokens[0]
        left, right = int(left), int(right)
        if op == '+':
            return left + right
        elif op == '-':
            return left - right
        elif op == '*':
            return left * right
        elif op == '/':
            return left / right
        return 0
    
    # Grammar with evaluation action
    number = pp.Word(pp.nums)
    operator = pp.oneOf("+ - * /")
    expression = pp.Group(number + operator + number).setParseAction(evaluate_expr)
    
    # Test expressions with immediate evaluation (measured: `/` is true division, so 20 / 4 evaluates to 5.0).
    # Every input is valid, so a ParseException fails the cell
    test_expressions = {"5 + 3": 8, "10 * 2": 20, "15 - 7": 8, "20 / 4": 5.0}
    
    for expr, expected in test_expressions.items():
        result = expression.parseString(expr)
        assert result[0] == expected, (expr, result[0])
        print(f"'{expr}' = {result[0]}")
    
    # Parse action for string processing
    def to_upper(tokens):
        return tokens[0].upper()
    
    word = pp.Word(pp.alphas).setParseAction(to_upper)
    sentence = word + pp.ZeroOrMore(word)
    
    result = sentence.parseString("hello world python")
    assert list(result) == ["HELLO", "WORLD", "PYTHON"], list(result)
    print(f"Uppercase transformation: {list(result)}")
    
    print("Parse actions and transformations completed successfully")
    
except ImportError as e:
    print(f"Import error: {e}")
except Exception as e:
    print(f"Runtime error: {e}")
    raise AssertionError(
        f"pyparsing example (cell 6) failed: {type(e).__name__}: {e}"
    ) from e

print("Example 3 finished")

## Example 4: Error handling and debugging


In [ ]:
import pyparsing as pp

# Example 4: Error handling and debugging
print("Example 4: Error handling and debugging")

try:
    # Define a strict grammar for email addresses
    username = pp.Word(pp.alphanums + "_.-")
    domain = pp.Word(pp.alphanums + "-") + pp.ZeroOrMore("." + pp.Word(pp.alphanums + "-"))
    email = username + "@" + domain
    
    # Test valid and invalid email addresses: (address, whether the grammar must accept it)
    test_emails = [
        ("user@example.com", True),      # Valid
        ("test.user@sub.domain.org", True),  # Valid
        ("invalid-email", False),         # Invalid - no @
        ("user@", False),                 # Invalid - no domain
        ("@domain.com", False),           # Invalid - no username
        ("user@domain", True),           # Valid but simple
    ]
    
    for test_email, should_parse in test_emails:
        try:
            result = email.parseString(test_email, parseAll=True)
        except pp.ParseException as e:
            assert not should_parse, f"valid address {test_email!r} was rejected: {e}"
            print(f"✗ Invalid email: {test_email} - Error at position {e.loc}: {e.msg}")
        else:
            assert should_parse, f"invalid address {test_email!r} was accepted: {list(result)}"
            print(f"✓ Valid email: {test_email}")
    
    # Demonstrate partial parsing vs full parsing
    partial_grammar = pp.Word(pp.nums)
    test_string = "123 abc"
    
    print(f"\nTesting partial vs full parsing with: '{test_string}'")
    
    # Partial parsing (default)
    partial_result = partial_grammar.parseString(test_string)
    assert list(partial_result) == ["123"], list(partial_result)
    print(f"Partial parse: {list(partial_result)}")
    
    # Full parsing (parseAll=True) must refuse the trailing " abc" (measured: at position 4)
    try:
        full_result = partial_grammar.parseString(test_string, parseAll=True)
    except pp.ParseException as e:
        assert (e.loc, e.msg) == (4, "Expected end of text"), (e.loc, e.msg)
        print(f"Full parse failed: {e.msg} at position {e.loc}")
    else:
        raise AssertionError(f"parseAll=True accepted trailing text in {test_string!r}: {list(full_result)}")
    
    print("Error handling and debugging completed successfully")
    
except ImportError as e:
    print(f"Import error: {e}")
except Exception as e:
    print(f"Runtime error: {e}")
    raise AssertionError(
        f"pyparsing example (cell 8) failed: {type(e).__name__}: {e}"
    ) from e

print("Example 4 finished")

## Example 5: Real-world configuration file parsing


In [ ]:
import pyparsing as pp

# Example 5: Real-world configuration file parsing
print("Example 5: Real-world configuration file parsing")

try:
    # Define grammar for INI-style configuration files
    # Comments start with # or ;
    comment = (pp.Literal("#") | pp.Literal(";")) + pp.restOfLine
    
    # Section headers [section_name]
    section_name = pp.Word(pp.alphanums + "_-")
    section_header = pp.Suppress("[") + section_name + pp.Suppress("]")
    
    # Key-value pairs: key = value
    key = pp.Word(pp.alphanums + "_-")
    value = pp.Regex(r"[^\r\n]*").setParseAction(lambda t: t[0].strip())
    key_value = pp.Group(key + pp.Suppress("=") + value)
    
    # Section content
    section_content = pp.ZeroOrMore(key_value | comment.suppress())
    section = pp.Group(section_header + section_content)
    
    # Complete config file
    config_file = pp.ZeroOrMore(section | comment.suppress())
    
    # Sample configuration text
    config_text = """
    # Database configuration
    [database]
    host = localhost
    port = 5432
    name = myapp_db
    user = dbuser
    
    ; Application settings
    [app]
    debug = true
    secret_key = your-secret-key-here
    max_connections = 100
    
    [logging]
    level = INFO
    file = /var/log/myapp.log
    """
    
    # Parse the configuration
    try:
        parsed_config = config_file.parseString(config_text)
        # measured: three sections, every key/value pair kept, values stripped
        assert parsed_config.asList() == [
            ["database", ["host", "localhost"], ["port", "5432"], ["name", "myapp_db"], ["user", "dbuser"]],
            ["app", ["debug", "true"], ["secret_key", "your-secret-key-here"], ["max_connections", "100"]],
            ["logging", ["level", "INFO"], ["file", "/var/log/myapp.log"]],
        ], parsed_config.asList()
        
        print("Parsed configuration:")
        for section in parsed_config:
            section_name = section[0]
            print(f"\n[{section_name}]")
            
            for item in section[1:]:
                if len(item) == 2:  # key-value pair
                    key, value = item
                    print(f"  {key} = {value}")
        
        # Extract specific values
        print(f"\nExtracting specific values:")
        for section in parsed_config:
            if section[0] == "database":
                for item in section[1:]:
                    if len(item) == 2 and item[0] == "host":
                        print(f"Database host: {item[1]}")
    
    except pp.ParseException as e:
        print(f"Configuration parse error: {e}")
        raise  # the sample configuration is valid: a parse error is a failure, not a demo
    
    print("Real-world configuration file parsing completed successfully")
    
except ImportError as e:
    print(f"Import error: {e}")
except Exception as e:
    print(f"Runtime error: {e}")
    raise AssertionError(
        f"pyparsing example (cell 10) failed: {type(e).__name__}: {e}"
    ) from e

print("Example 5 finished")